# Gemelo digital — Control de calidad de botellas (Teachable Machine -> Ubidots)

Este notebook toma el modelo que entrenaste en **Teachable Machine** (botella *con tapa* = OK / *sin tapa* = Defectuoso),
clasifica en tiempo real lo que ve la **webcam** y envia el resultado a un **dashboard de Ubidots**.

Variables que se envian al gemelo digital:
- `estado` -> 1 = Producto OK (con tapa) | 0 = Defectuoso (sin tapa)
- `confianza` -> % de confianza de la prediccion
- `porcentaje_defectuosos` -> % acumulado de defectuosos detectados (para alertas)

---


## 1. Exportar el modelo desde Teachable Machine

En tu proyecto de Teachable Machine:

1. Pulsa **Export Model** (arriba a la derecha).
2. Pestaña **Tensorflow** (NO Tensorflow.js ni TFLite).
3. Formato **Keras** -> **Download my model**.
4. Se descarga un `.zip` con dos archivos: **`keras_model.h5`** y **`labels.txt`**.
5. Descomprime y deja esos dos archivos **en la misma carpeta que este notebook**.


## 2. Instalar dependencias  *(IMPORTANTE: leer)*

El `.h5` de Teachable Machine esta guardado en formato **Keras 2**. Con las versiones nuevas
(TensorFlow >= 2.16, que trae Keras 3) **falla al cargar** con un error de deserializacion.

La forma mas fiable de evitarlo es usar **Python 3.10 o 3.11** y fijar **TensorFlow 2.12**, que
trae Keras 2 y carga el modelo sin tocar nada.

Ejecuta la siguiente celda **una sola vez**. Si ya tienes el entorno montado, saltala.


In [1]:
# Ejecutar UNA sola vez. Recomendado: Python 3.10 o 3.11.
%pip install "tensorflow==2.12.1" opencv-python pillow numpy requests

INFO: pip is looking at multiple versions of grpcio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cryptography to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cryptography to determine which version is compatible with other requirements. This could take a while.
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/272.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/272.8 MB ? eta -:--:--
   ---------------------------------------- 0.8/272.8 MB 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
exceptiongroup 1.3.1 requires typing-extensions>=4.6.0; python_version < "3.13", but you have typing-extensions 4.5.0 which is incompatible.
ipython 8.39.0 requires typing_extensions>=4.6; python_version < "3.12", but you have typing-extensions 4.5.0 which is incompatible.
mediapipe 0.10.9 requires protobuf<4,>=3.11, but you have protobuf 4.25.9 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.3 which is incompatible.
optree 0.19.0 requires typing-extensions>=4.6.0, but you have typing-extensions 4.5.0 which is incompatible.
tf-keras 2.20.1 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.12.1 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --up

**Alternativa** si estas obligado a usar TensorFlow nuevo (>= 2.16) y no puedes instalar el 2.12:

```bash
pip install tf-keras opencv-python pillow numpy requests
```

y AÑADE estas dos lineas **al principio** de la celda de imports (antes de importar keras):

```python
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
```

Eso fuerza a usar Keras 2 (paquete `tf-keras`) y el `.h5` vuelve a cargar.


## 3. Configuracion

Ajusta aqui tu token, el dispositivo y, sobre todo, **cual clase es la OK** segun tu `labels.txt`.

In [31]:
# Si usas la alternativa con TensorFlow >= 2.16, descomenta estas 2 lineas:
# import os
# os.environ["TF_USE_LEGACY_KERAS"] = "1"
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# ---- Credenciales y dispositivo de Ubidots ----
TOKEN        = "BBUS-kO9PBDeuCP9JgUvEXLVIgw4nz3v5xS"  # tu token de Ubidots
DEVICE_LABEL = "Cinta_Transportadora_01"              # etiqueta del dispositivo

# ---- Etiquetas de las variables en Ubidots ----
VAR_ESTADO      = "estado"                  # 1 = OK / 0 = defectuoso
VAR_CONFIANZA   = "confianza"               # % de confianza de la prediccion
VAR_PORCENTAJE  = "porcentaje_defectuosos"  # % acumulado de defectuosos

# ---- Archivos del modelo de Teachable Machine ----
MODELO_PATH = "keras_model.h5"
LABELS_PATH = "labels.txt"

MODELO_PATH = r"C:\Users\GLEN\Desktop\Model\keras_model.h5"
LABELS_PATH = r"C:\Users\GLEN\Desktop\Model\labels.txt"

# ---- Cual de tus clases significa "OK" (con tapa) ----
# Mira tu labels.txt. TM lo guarda como:  "0 Con tapa" / "1 Sin tapa"  (o como las nombraste).
# Pon aqui el INDICE de la clase que representa el producto BUENO (con tapa).
INDICE_CLASE_OK = 0

# ---- Comportamiento del envio ----
INTERVALO_ENVIO = 2.0   # segundos entre cada envio a Ubidots (evita saturar la API)
UMBRAL_CONFIANZA = 0.60 # por debajo de esto se considera prediccion dudosa (no se cuenta)
CAMARA_ID = 0           # 0 = webcam por defecto

## 4. Cargar el modelo y las etiquetas

In [32]:
import numpy as np
from keras.models import load_model

np.set_printoptions(suppress=True)

# compile=False: solo queremos inferencia, no reentrenar
model = load_model(MODELO_PATH, compile=False)

# labels.txt -> ["Con tapa", "Sin tapa", ...] quitando el numero del inicio
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    class_names = [linea.strip().split(" ", 1)[-1] for linea in f.readlines() if linea.strip()]

print("Modelo cargado correctamente.")
print("Clases detectadas:")
for i, nombre in enumerate(class_names):
    marca = "  <-- OK (con tapa)" if i == INDICE_CLASE_OK else ""
    print(f"  [{i}] {nombre}{marca}")
print("\nSi el indice OK no coincide con tu botella CON TAPA, corrige INDICE_CLASE_OK arriba.")

Modelo cargado correctamente.
Clases detectadas:
  [0] Botella OK  <-- OK (con tapa)
  [1] Botella not ok

Si el indice OK no coincide con tu botella CON TAPA, corrige INDICE_CLASE_OK arriba.


## 5. Funciones auxiliares

- `preprocesar`: deja la imagen como espera Teachable Machine (224x224, normalizada a [-1, 1]).
- `clasificar`: devuelve indice de clase, nombre y confianza.
- `build_payload` y `post_request`: adaptadas de tu script `digital_twin.py`, pero ahora con
  datos REALES del modelo y usando HTTPS.


In [33]:
import cv2

def preprocesar(frame):
    """Convierte un frame BGR de OpenCV al tensor que espera el modelo de TM."""
    img = cv2.resize(frame, (224, 224), interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)              # TM espera RGB
    img = np.asarray(img, dtype=np.float32).reshape(1, 224, 224, 3)
    img = (img / 127.5) - 1                                 # normalizacion de TM a [-1, 1]
    return img

def clasificar(frame):
    """Devuelve (indice, nombre_clase, confianza_0a1)."""
    pred = model.predict(preprocesar(frame), verbose=0)
    indice = int(np.argmax(pred))
    confianza = float(pred[0][indice])
    return indice, class_names[indice], confianza

In [34]:
import time
import requests

URL = f"https://industrial.api.ubidots.com/api/v1.6/devices/{DEVICE_LABEL}"

def build_payload(estado, confianza_pct, porcentaje_def, etiqueta):
    """Arma el payload para Ubidots. Adaptado de tu digital_twin.py.
    Se adjunta la etiqueta de texto como 'context' para verla en el dashboard."""
    payload = {
        VAR_ESTADO:     {"value": estado, "context": {"clase": etiqueta}},
        VAR_CONFIANZA:  confianza_pct,
        VAR_PORCENTAJE: porcentaje_def,
    }
    return payload

def post_request(payload):
    """Envia el payload a Ubidots con reintentos (igual logica que tu script original)."""
    headers = {"X-Auth-Token": TOKEN, "Content-Type": "application/json"}
    status = 400
    intentos = 0
    while status >= 400 and intentos <= 5:
        try:
            req = requests.post(url=URL, headers=headers, json=payload, timeout=10)
            status = req.status_code
        except requests.exceptions.RequestException as e:
            print(f"[ERROR] Conexion: {e}")
            status = 400
        intentos += 1
        if status >= 400:
            time.sleep(1)
    if status >= 400:
        print("[ERROR] No se pudo enviar tras 5 intentos. Revisa token e internet.")
        return False
    return True

## 6. Bucle principal — clasificacion en vivo + envio al gemelo digital

Abre una ventana de OpenCV con la webcam. En cada frame:
1. Clasifica (OK / Defectuoso) y muestra el resultado en pantalla.
2. Cada `INTERVALO_ENVIO` segundos envia el estado a Ubidots y actualiza el % de defectuosos.

**Pulsa la tecla `q` sobre la ventana de video para detener** (no uses el boton de parar de Jupyter,
para que la camara se libere bien).


In [35]:
camara = cv2.VideoCapture(CAMARA_ID)

total_piezas = 0
total_defectuosos = 0
ultimo_envio = 0.0

print("Camara iniciada. Pulsa 'q' sobre la ventana de video para salir.")

try:
    while True:
        ok, frame = camara.read()
        if not ok:
            print("[ERROR] No se pudo leer la camara.")
            break

        indice, etiqueta, confianza = clasificar(frame)
        es_ok = (indice == INDICE_CLASE_OK)

        # Texto sobre el video
        estado_txt = "OK (con tapa)" if es_ok else "DEFECTUOSO (sin tapa)"
        color = (0, 200, 0) if es_ok else (0, 0, 255)
        cv2.putText(frame, f"{estado_txt}  {confianza*100:.0f}%",
                    (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        cv2.imshow("Control de calidad - pulsa q para salir", frame)

        # Envio periodico a Ubidots (solo si la prediccion es suficientemente segura)
        ahora = time.time()
        if confianza >= UMBRAL_CONFIANZA and (ahora - ultimo_envio) >= INTERVALO_ENVIO:
            total_piezas += 1
            if not es_ok:
                total_defectuosos += 1
            porcentaje_def = round(100 * total_defectuosos / total_piezas, 1)
            estado_val = 1 if es_ok else 0

            payload = build_payload(estado_val, round(confianza * 100, 1),
                                    porcentaje_def, etiqueta)
            if post_request(payload):
                print(f"[OK] enviado -> estado={estado_val} ({etiqueta}) "
                      f"conf={confianza*100:.0f}% def={porcentaje_def}%")
            ultimo_envio = ahora

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    camara.release()
    cv2.destroyAllWindows()
    print(f"\nFin. Piezas analizadas: {total_piezas} | Defectuosas: {total_defectuosos}")

Camara iniciada. Pulsa 'q' sobre la ventana de video para salir.
[OK] enviado -> estado=0 (Botella not ok) conf=96% def=100.0%
[OK] enviado -> estado=1 (Botella OK) conf=88% def=50.0%
[OK] enviado -> estado=0 (Botella not ok) conf=75% def=66.7%
[OK] enviado -> estado=0 (Botella not ok) conf=92% def=75.0%
[OK] enviado -> estado=0 (Botella not ok) conf=79% def=80.0%
[OK] enviado -> estado=1 (Botella OK) conf=65% def=66.7%
[OK] enviado -> estado=1 (Botella OK) conf=90% def=57.1%
[OK] enviado -> estado=1 (Botella OK) conf=94% def=50.0%
[OK] enviado -> estado=1 (Botella OK) conf=65% def=44.4%
[OK] enviado -> estado=1 (Botella OK) conf=100% def=40.0%
[OK] enviado -> estado=0 (Botella not ok) conf=77% def=45.5%
[OK] enviado -> estado=0 (Botella not ok) conf=76% def=50.0%
[OK] enviado -> estado=1 (Botella OK) conf=71% def=46.2%
[OK] enviado -> estado=1 (Botella OK) conf=76% def=42.9%
[OK] enviado -> estado=1 (Botella OK) conf=93% def=40.0%
[OK] enviado -> estado=0 (Botella not ok) conf=96% def

## 7. En Ubidots (dashboard)

Al ejecutar el bucle, el dispositivo `Cinta_Transportadora_01` creara automaticamente las 3 variables
si no existen. Despues, en **Dashboards -> Add Widget**, puedes montar:

- **Indicador / Metric** sobre `estado` -> verde si 1, rojo si 0.
- **Line chart** de `porcentaje_defectuosos` -> ver picos de defectos en el tiempo.
- **Gauge** de `confianza`.
- **Alert (Events)**: `porcentaje_defectuosos` > 20% -> email/Telegram. Esto cubre el punto del
  enunciado de *"alertas automaticas cuando el porcentaje defectuoso supere cierto umbral"*.

## 8. Solucion de problemas

- **El `.h5` no carga / error de deserializacion** -> tienes Keras 3. Usa TensorFlow 2.12 (celda 2)
  o la alternativa con `tf-keras` + `TF_USE_LEGACY_KERAS=1`.
- **La ventana de la camara no abre o se queda colgada** -> ejecuta el notebook en tu maquina local
  (no en un Jupyter remoto / Colab; alli `cv2.imshow` no funciona).
- **Siempre clasifica al reves** -> cambia `INDICE_CLASE_OK` (0 <-> 1).
- **Error 401 de Ubidots** -> token incorrecto. **Error 403** -> el token no tiene permisos de escritura.
